In [294]:

import pandas as pd
import numpy as np

### Function to take out columns with no data, data less than 25% filled, and specific named columns

In [295]:
def clean_csv(input_path, output_path, threshold=0.25, drop_cols=None,  keep_cols=None):
    df = pd.read_csv(input_path, dtype=str).replace(r"^\s*$", pd.NA, regex=True)

    sparse  = df.columns[df.notna().mean() < threshold].tolist()
    sparse  = [c for c in sparse if c not in (keep_cols or [])]  
    named   = [c for c in (drop_cols or []) if c in df.columns]

    df.drop(columns=list(dict.fromkeys(sparse + named)), inplace=True)
    df.to_csv(output_path, index=False)

    print(f"\n{input_path} → {output_path} | {len(df.columns)} columns kept")
    print(f"  Dropped sparse : {sparse or 'none'}")
    print(f"  Dropped named  : {named or 'none'}")
    
    return df

#### Calling Cleaning Function

In [296]:
clean_capacity = clean_csv("data/capacity.csv", "data/capacity_new.csv", threshold = 0.25, drop_cols = ["source_id","comment"], keep_cols = ["material"])
clean_coal = clean_csv("data/coal.csv", "data/coal_new.csv", threshold = 0.25, drop_cols = ["source_id","comment", "amount_sold_tonnes"])
clean_commodities = clean_csv("data/commodities.csv", "data/commodoties_new.csv", threshold = 0.25, drop_cols = ["source_id","comment", "amount_sold_tonnes","metal_payable_tonnes","mine_processing", "id"])
clean_minerals = clean_csv("data/minerals.csv", "data/minerals_new.csv", threshold = 0.25, drop_cols = ["source_id","comment", "amount_sold_tonnes","mine_processing"])
clean_reserves = clean_csv("data/reserves.csv", "data/reserves_new.csv", threshold = 0.25, drop_cols = ["source_id","comment"])
clean_waste = clean_csv("data/waste.csv", "data/waste_new.csv", threshold = 0.25, drop_cols = ["source_id","comment"], keep_cols = ["material"])


data/capacity.csv → data/capacity_new.csv | 6 columns kept
  Dropped sparse : ['comment']
  Dropped named  : ['source_id', 'comment']

data/coal.csv → data/coal_new.csv | 6 columns kept
  Dropped sparse : ['amount_sold_tonnes', 'reporting_period', 'comment']
  Dropped named  : ['source_id', 'comment', 'amount_sold_tonnes']

data/commodities.csv → data/commodoties_new.csv | 8 columns kept
  Dropped sparse : ['yield_ppm', 'amount_sold_tonnes', 'metal_payable_tonnes', 'mine_processing', 'reporting_period', 'comment']
  Dropped named  : ['source_id', 'comment', 'amount_sold_tonnes', 'metal_payable_tonnes', 'mine_processing', 'id']

data/minerals.csv → data/minerals_new.csv | 6 columns kept
  Dropped sparse : ['overall_grade_ppm', 'amount_sold_tonnes', 'mine_processing', 'reporting_period', 'comment']
  Dropped named  : ['source_id', 'comment', 'amount_sold_tonnes', 'mine_processing']

data/reserves.csv → data/reserves_new.csv | 8 columns kept
  Dropped sparse : ['comment']
  Dropped named

### Changing strings to be numbers so we can do calculations (i.e. reserve back/forward filling)

In [297]:
# reserves
clean_reserves["mineral_value_tonnes"]   = pd.to_numeric(clean_reserves["mineral_value_tonnes"], errors="coerce")
clean_reserves["commodity_value_tonnes"] = pd.to_numeric(clean_reserves["commodity_value_tonnes"], errors="coerce")
clean_reserves["grade_ppm"]              = pd.to_numeric(clean_reserves["grade_ppm"], errors="coerce")

# coal
clean_coal["value_tonnes"]               = pd.to_numeric(clean_coal["value_tonnes"], errors="coerce")

# minerals
clean_minerals["value_tonnes"]           = pd.to_numeric(clean_minerals["value_tonnes"], errors="coerce")

# commodities
clean_commodities["value_tonnes"]        = pd.to_numeric(clean_commodities["value_tonnes"], errors="coerce")
clean_commodities["grade_ppm"]           = pd.to_numeric(clean_commodities["grade_ppm"], errors="coerce")
clean_commodities["recovery_rate"]       = pd.to_numeric(clean_commodities["recovery_rate"], errors="coerce")

# waste
clean_waste["value_tonnes"]              = pd.to_numeric(clean_waste["value_tonnes"], errors="coerce")
clean_waste["total_material_tonnes"]     = pd.to_numeric(clean_waste["total_material_tonnes"], errors="coerce")

# capacity
clean_capacity["value_tpa"]              = pd.to_numeric(clean_capacity["value_tpa"], errors="coerce")

# Convert year to integer in all clean files
for df in [clean_capacity, clean_coal, clean_minerals, clean_reserves, clean_waste, clean_commodities]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

print("Done!")
# Want to be using filename_new after running this cell for future uses

Done!


### Function for Back/Forward Filling of Reserve Data Values

In [298]:
output_path   = "data/reserves_filled.csv"

# Add together coal and mineral production files
production = pd.concat([
    clean_coal[clean_coal["type"] == "Coal mined"][["facility_id", "year", "material", "value_tonnes"]],
    clean_minerals[clean_minerals["type"] == "Ore mined"][["facility_id", "year", "material", "value_tonnes"]]
], ignore_index=True)

# Backfill and forward fill from reserve values given
# If 2 values given, assume new prospecting at later date, and only forward fill from this point
def fill_one_group(known_rows, production):
    fid, mat   = known_rows["facility_id"].iloc[0], known_rows["material"].iloc[0]
    anchors    = dict(zip(known_rows["year"], known_rows["mineral_value_tonnes"]))
    prod       = production[(production["facility_id"] == fid) & (production["material"] == mat)]
    prod       = prod.set_index("year")["value_tonnes"].to_dict()
    if not prod:
        return pd.DataFrame()

    anchor_years = sorted(anchors)
    estimates    = {}

    for i, ay in enumerate(anchor_years):
        av          = anchors[ay]
        next_anchor = anchor_years[i + 1] if i + 1 < len(anchor_years) else None
        if pd.isna(av):
            continue

        val = av
        for y in sorted(y for y in prod if y > ay):
            if next_anchor and y >= next_anchor:
                break
            val -= prod[y]
            estimates[y] = val

        if i == 0:
            val = av
            for y in sorted((y for y in prod if y < ay), reverse=True):
                val += prod[y]
                estimates[y] = val

    template = known_rows.iloc[0].to_dict()
    return pd.DataFrame([
        {**template, "year": y, "mineral_value_tonnes": v,
         "commodity_value_tonnes": None, "grade_ppm": None,
         "source_id": "filled", "comment": None}
        for y, v in estimates.items() if y not in anchors
    ])

#### Run Code to get new reserve values

In [299]:
# Run the code and recieve new CSV file
reserves   = clean_reserves

filled_parts = [fill_one_group(g, production)
                for _, g in reserves.groupby(["facility_id", "material"])]
filled_parts = [f for f in filled_parts if not f.empty]

all_filled = pd.concat(filled_parts, ignore_index=True) if filled_parts else pd.DataFrame()
result     = pd.concat([reserves, all_filled], ignore_index=True) if not all_filled.empty else reserves.copy()

result.sort_values(["facility_id", "material", "year"], inplace=True)
result.to_csv(output_path, index=False)

print(f"Original rows : {len(reserves)}")
print(f"Filled rows   : {len(all_filled)}")

Original rows : 1039
Filled rows   : 4476


### Code to Remove Rows of Data found by ID in ids_no_start_date

In [300]:
#Still need to make sure that the variable id_no_start_date is in the right place for this to run

%run "data_processing_final2.ipynb"
print("this is before", len(clean_capacity))
def remove_no_start_date(df, ids_to_remove):
    return df[~df["facility_id"].isin(ids_to_remove)]

clean_capacity    = remove_no_start_date(clean_capacity, ids_no_start_date)
clean_coal        = remove_no_start_date(clean_coal, ids_no_start_date)
clean_commodities = remove_no_start_date(clean_commodities, ids_no_start_date)
clean_minerals    = remove_no_start_date(clean_minerals, ids_no_start_date)
clean_reserves    = remove_no_start_date(clean_reserves, ids_no_start_date)
clean_waste       = remove_no_start_date(clean_waste, ids_no_start_date)
print("this is after",len(clean_capacity))

/Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project
Layer: facilities
Columns:
column name: facility_id data type: str
column name: facility_name data type: str
column name: facility_other_names data type: str
column name: sub_site_name data type: object
column name: sub_site_other_names data type: object
column name: facility_type data type: str
column name: primary_commodity data type: str
column name: commodities_products data type: str
column name: facility_equipment data type: str
column name: production_start data type: float64
column name: production_end data type: float64
column name: activity_status data type: str
column name: activity_status_year data type: float64
column name: surface_area_sq_km data type: float64
column name: concession_area_sq_km data type: float64
column name: country data type: str
column name: GID_0 data type: str
column name: GID_1 data type: str
column name: GID_2 data type: str
column name: GID_3 data type: str
column name: GID_4 data typ

In [301]:
#pip install nbformat
%run data_processing_final2.ipynb
print(ids_no_start_date)


/Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project
Layer: facilities
Columns:
column name: facility_id data type: str
column name: facility_name data type: str
column name: facility_other_names data type: str
column name: sub_site_name data type: object
column name: sub_site_other_names data type: object
column name: facility_type data type: str
column name: primary_commodity data type: str
column name: commodities_products data type: str
column name: facility_equipment data type: str
column name: production_start data type: float64
column name: production_end data type: float64
column name: activity_status data type: str
column name: activity_status_year data type: float64
column name: surface_area_sq_km data type: float64
column name: concession_area_sq_km data type: float64
column name: country data type: str
column name: GID_0 data type: str
column name: GID_1 data type: str
column name: GID_2 data type: str
column name: GID_3 data type: str
column name: GID_4 data typ

## Function to merge all of the .csv files together including the facilities data and the price data

In [302]:
COMMODITY_PRICE_MAP = {
    "Coal":                  "Coal",
    "Nickel":                "Nickel",
    "Gold":                  "Gold",
    "Copper":                "Copper",
    "Aluminium":             "Aluminium",
    "Silver":                "Silver",
    "Zinc":                  "Zinc",
    "Iron":                  "Iron",
    "Other (poly)-metallic": "Other",
    "Other mine":            "Other",
}

KEY = ["facility_id", "year", "material"]

def prep_capacity(df):
    # has: facility_id, year, material, commodity, value_tpa
    return df.rename(columns={"value_tpa": "capacity_value_tpa"})

def prep_coal(df):
    # has: facility_id, year, type, material, value_tonnes
    pivoted = df.pivot_table(
        index=KEY, columns="type", values="value_tonnes", aggfunc="sum"
    ).reset_index()
    pivoted.columns.name = None
    return pivoted.rename(columns={
        "Coal mined": "coal_mined_value_tonnes",
    })

def prep_minerals(df):
    # has: facility_id, year, type, material, value_tonnes
    pivoted = df.pivot_table(
        index=KEY, columns="type", values="value_tonnes", aggfunc="sum"
    ).reset_index()
    pivoted.columns.name = None
    return pivoted.rename(columns={
        "Ore mined":     "minerals_ore_mined_value_tonnes",
    })

def prep_reserves(df):
    # has: facility_id, year, material, commodity, mineral_value_tonnes,
    #      commodity_value_tonnes, grade_ppm, source_id
    return df.drop(columns=["source_id"], errors="ignore").rename(columns={
        "mineral_value_tonnes":   "reserves_mineral_value_tonnes",
        "commodity_value_tonnes": "reserves_commodity_value_tonnes",
        "grade_ppm":              "reserves_grade_ppm",
    })

def prep_waste(df):
    # has: facility_id, year, waste_type, value_tonnes, total_material_tonnes
    # (type_mining, material, commodity, stripping_ratio were dropped as sparse)
    return df.rename(columns={
        "value_tonnes":          "waste_value_tonnes",
        "total_material_tonnes": "waste_total_material_tonnes",
    })

def prep_commodities(df):
    # has: facility_id, year, material, commodity, value_tonnes, grade_ppm, recovery_rate, id_minerals
    return df.rename(columns={
        "value_tonnes":   "commodities_value_tonnes",
        "grade_ppm":      "commodities_grade_ppm",
        "recovery_rate":  "commodities_recovery_rate",
        "id_minerals":    "commodities_id_minerals",
    })

def prep_price(price_df, facilities_df):
    price_long = price_df.melt(
        id_vars="Year", var_name="price_commodity", value_name="price_usd_tonne"
    ).rename(columns={"Year": "year"})
    fac_lookup = facilities_df[["facility_id", "primary_commodity"]].copy()
    fac_lookup["price_commodity"] = fac_lookup["primary_commodity"].map(COMMODITY_PRICE_MAP)
    return fac_lookup.merge(price_long, on="price_commodity", how="left")[
        ["facility_id", "year", "price_usd_tonne"]
    ]


### Running Mothership Merge code

In [303]:
#run above functions to rename important columns and merge everything
facilities_df = pd.read_csv("data/final_facilities.csv")
price_df      = pd.read_csv("data/price_data.csv")
price         = prep_price(price_df, facilities_df)

capacity    = prep_capacity(clean_capacity.copy())
coal        = prep_coal(clean_coal.copy())
minerals    = prep_minerals(clean_minerals.copy())
reserves    = prep_reserves(result.copy())   # result = your filled reserves dataframe
waste       = prep_waste(clean_waste.copy())
commodities = prep_commodities(clean_commodities.copy())

# Build base from union of all facility-year-material keys
all_keys = pd.concat([
    df[KEY] for df in [capacity, coal, minerals, reserves, waste, commodities]
]).drop_duplicates().reset_index(drop=True)

# Merge each file on facility_id + year + material
merged = all_keys
for name, df in [("capacity", capacity), ("coal", coal), ("minerals", minerals),
                 ("reserves", reserves), ("waste", waste), ("commodities", commodities)]:
    merged = merged.merge(df, on=KEY, how="left")
    print(f"Merged {name}: {len(df)} rows → merged now has {len(merged.columns)} columns")

# Join facilities static info on facility_id only
merged = merged.merge(
    facilities_df[["facility_id", "facility_name", "facility_type", "primary_commodity",
                   "commodities_products", "facility_equipment",
                    "country","LOM_in_Buckets","still_operating"]],
    on="facility_id", how="left"
)

# Join price on facility_id + year
merged = merged.merge(price, on=["facility_id", "year"], how="left")

mother_file = merged.sort_values(KEY, inplace=False)  # renamed from merged

drop_cols_og = ['Clean coal']

mother_file = mother_file.drop(columns = drop_cols_og)
mother_file.to_csv("data/mother_file.csv", index=False)

print(f"\nFinal shape : {mother_file.shape}")
print(f"Output saved: data/mother_file.csv")

Merged capacity: 59 rows → merged now has 6 columns
Merged coal: 1733 rows → merged now has 8 columns
Merged minerals: 2673 rows → merged now has 11 columns
Merged reserves: 5515 rows → merged now has 17 columns
Merged waste: 496 rows → merged now has 21 columns
Merged commodities: 3867 rows → merged now has 26 columns

Final shape : (10841, 34)
Output saved: data/mother_file.csv


In [ ]:
# Transform mother_file.csv to have each facility_id once with year-based columns
import pandas as pd

mother = pd.read_csv('data/mother_file.csv')

drop_cols = ['facility_name', 'facility_type', 'commodities_products', 'facility_equipment',
               'material', 'commodity_y', 'commodities_id_minerals', 'commodity', 'id_y','waste_type',
               'capacity_value_tpa','id','id_x','commodity_x']

mother = mother.drop(columns = drop_cols)

# Define static columns (not pivoted by year)
static_cols = [ 'primary_commodity', 'country', 'LOM_in_Buckets', 
               'still_operating', 'price_usd_tonne']

# Varying columns (pivoted by year)
varying_cols = [col for col in mother.columns if col not in ['facility_id', 'year'] and col not in static_cols]

# Create static DataFrame (one row per facility_id)
static_df = mother[['facility_id'] + static_cols].drop_duplicates('facility_id')

# Create varying DataFrame and pivot
varying_df = mother[['facility_id', 'year'] + varying_cols]
varying_pivot = varying_df.pivot_table(index='facility_id', columns='year', values=varying_cols, aggfunc='first')

# Flatten the multi-level columns
varying_pivot.columns = [f'{year}_{col}' for col, year in varying_pivot.columns]

# Reset index
varying_pivot.reset_index(inplace=True)

# Merge static and varying
mother2 = static_df.merge(varying_pivot, on='facility_id', how='left')

# Sort columns by column name then by year
other_cols = [col for col in mother2.columns if col != 'facility_id' and col not in static_cols]
sorted_cols = sorted(other_cols, key=lambda x: (x.split('_', 1)[1], int(x.split('_')[0])))
mother2 = mother2[['facility_id'] + static_cols + sorted_cols]

# Remove rows without LOM_in_Buckets
mother2 = mother2.dropna(subset=['LOM_in_Buckets'])

# Save to new CSV
mother2.to_csv('data/mother2.csv', index=False)

print("mother2.csv created with static columns and pivoted varying columns.")
print(f"Shape: {mother2.shape}")
print(mother2.head())

KeyError: "['Clean coal'] not found in axis"

### Beginnings of adding salary data

In [ ]:
# Add mining salary per country to mother2
import pandas as pd

mother_with_salaries = pd.read_csv('data/mother2.csv')
wages = pd.read_excel('data/wage_per_country.xlsx')

# Normalize column names and country text
wages.columns = [c.strip().lower() for c in wages.columns]
if 'country' not in wages.columns or len(wages.columns) < 2:
    raise ValueError('Expected wage_per_country.xlsx to have country and salary columns')

salary_col = [c for c in wages.columns if c != 'country'][0]
wages = wages.rename(columns={salary_col: 'mining_salary'})

mother_with_salaries['country'] = mother_with_salaries['country'].astype(str).str.strip()
wages['country'] = wages['country'].astype(str).str.strip()

mother_with_salaries = mother_with_salaries.merge(wages[['country', 'mining_salary']], on='country', how='left')

missing_count = mother_with_salaries['mining_salary'].isna().sum()
print(f"Merged mining salaries into mother2: {missing_count} rows missing salary")

mother_with_salaries.to_csv('data/mother_with_salaries.csv', index=False)
print('Saved updated mother_with_salaries.csv with mining_salary column')

Merged mining salaries into mother2: 0 rows missing salary
Saved updated mother_with_salaries.csv with mining_salary column


In [ ]:
# Move 'mining_salary' column after 'country' in mother_with_salaries
def move_column_after(df, col_to_move, after_col):
    cols = list(df.columns)
    cols.remove(col_to_move)
    insert_at = cols.index(after_col) + 1
    cols.insert(insert_at, col_to_move)
    return df[cols]

mother_with_salaries = move_column_after(mother_with_salaries, 'mining_salary', 'country')

# Save the updated DataFrame
mother_with_salaries.to_csv('data/mother_with_salaries.csv', index=False)
print("Moved 'mining_salary' after 'country' and saved updated mother_with_salaries.csv")

Moved 'mining_salary' after 'country' and saved updated mother_with_salaries.csv


In [ ]:
mother_with_salaries['country'].unique()

<StringArray>
['United States of America',                    'Spain',
                    'Ghana',               'Kazakhstan',
             'Saudi Arabia',                   'Mexico',
       'Russian Federation',                 'Portugal',
               'Madagascar',                    'Chile',
                   'Brazil',                     'Peru',
                    'China',                'Australia',
                'Indonesia',             'South Africa',
               'Mozambique',                   'Canada',
                    'India',               'Kyrgyzstan',
                   'Guinea',                 'Colombia',
                'Argentina',                   'Zambia',
                 'Suriname',              'Philippines',
                'Guatemala',                 'DR Congo',
                  'Ukraine',                  'Jamaica',
                  'Finland',                     'Mali',
                   'Poland',              'New Zealand',
                 

In [ ]:
## Adding On-Hot encooding for country and primary commodity
from sklearn.preprocessing import OneHotEncoder

country_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
commodity_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

final_df = pd.read_csv('data/mother_with_salaries.csv')
encoded_countries = country_encoder.fit_transform(final_df[['country']])
encoded_commodities = commodity_encoder.fit_transform(final_df[['primary_commodity']])
encoded_country_df = pd.DataFrame(encoded_countries, columns=country_encoder.get_feature_names_out())
encoded_commodity_df = pd.DataFrame(encoded_commodities, columns=commodity_encoder.get_feature_names_out())
final_df = pd.concat([final_df, encoded_country_df, encoded_commodity_df], axis=1)
final_df.to_csv('data/one_hot_mama.csv', index=False)
print('Saved one_hot_mama.csv with one-hot encoded')

Saved one_hot_mama.csv with one-hot encoded
